In [1]:
import pandas as pd
#opening the data file in read mode
df = pd.read_csv('Sentiment Analysis Dataset.tsv', sep='\t')
print(df)

            id  sentiment                                             review
0       5814_8          1  With all this stuff going down at the moment w...
1       2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2       7759_3          0  The film starts with a manager (Nicholas Bell)...
3       3630_4          0  It must be assumed that those who praised this...
4       9495_8          1  Superbly trashy and wondrously unpretentious 8...
...        ...        ...                                                ...
24995   3453_3          0  It seems like more consideration has gone into...
24996   5064_1          0  I don't believe they made this film. Completel...
24997  10905_3          0  Guy is a loser. Can't get girls, needs to buil...
24998  10194_3          0  This 30 minute documentary Buñuel made in the ...
24999   8478_8          1  I saw this movie as a child and it broke my he...

[25000 rows x 3 columns]


In [2]:
reviews=df.iloc[:,2].values
labels=df.iloc[:,1].values


In [3]:
#parsing html
from bs4 import BeautifulSoup
import re

def parseHtml(html):
  soup = BeautifulSoup(html, 'html.parser')
  return soup.get_text()

def removeDigits(string):
  for i in range(10):
    string=string.replace(str(i),' ')
  return string

#removing html
reviews=list(map(parseHtml, reviews))

#removing digits
reviews=list(map(removeDigits, reviews))


In [4]:
#tokenizing
import nltk
nltk.download('punkt')
tokenizedText=[nltk.word_tokenize(item) for item in reviews]


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     Missing Authority Key Identifier (_ssl.c:1032)>


In [5]:
#removing punctuation
punc = '''!()-[]{};:'"\, <>./?@#$%^&*_~'''
tokenizedText= [[word for word in review if word not in punc] for review in tokenizedText]


<>:2: SyntaxWarning: invalid escape sequence '\,'
<>:2: SyntaxWarning: invalid escape sequence '\,'
C:\Users\kesh2\AppData\Local\Temp\ipykernel_4200\404723166.py:2: SyntaxWarning: invalid escape sequence '\,'
  punc = '''!()-[]{};:'"\, <>./?@#$%^&*_~'''


In [6]:
import numpy as np
#splitting the Dataset into train and test set
totalRows=len(tokenizedText)

splitRatio=0.75
splitPoint=int(splitRatio*totalRows)

trainReviews=tokenizedText[:splitPoint]
trainLabels=labels[:splitPoint]
testReviews=tokenizedText[splitPoint:]
testLabels=labels[splitPoint:]

In [7]:
#learning word embeddings on training data using Gensim library
from gensim.models import Word2Vec, KeyedVectors
import nltk

embeddingsSize=128

# Train multiple Word2Vec models with different context window sizes
print("Training Word2Vec models with different window sizes...")

model_w3 = Word2Vec(trainReviews, vector_size=embeddingsSize, window=3, min_count=1, workers=4)
print("✓ Word2Vec model with window=3 trained")

model_w5 = Word2Vec(trainReviews, vector_size=embeddingsSize, window=5, min_count=1, workers=4)
print("✓ Word2Vec model with window=5 trained")

model_w7 = Word2Vec(trainReviews, vector_size=embeddingsSize, window=7, min_count=1, workers=4)
print("✓ Word2Vec model with window=7 trained")

######################Training of Word Embeddings Vectors Completed#########

Training Word2Vec models with different window sizes...
✓ Word2Vec model with window=3 trained
✓ Word2Vec model with window=5 trained
✓ Word2Vec model with window=7 trained


In [8]:
import numpy as np

def getVectors(dataset, model):
  """Get average word vectors for each review using a specific Word2Vec model"""
  vectors = []
  for dataItem in dataset:
    singleDataItemEmbedding = np.zeros(embeddingsSize)
    wordCount = 0
    for word in dataItem:
      if word in model.wv.key_to_index:
        singleDataItemEmbedding = singleDataItemEmbedding + model.wv[word]
        wordCount = wordCount + 1
    
    if wordCount > 0:
      singleDataItemEmbedding = singleDataItemEmbedding / wordCount
    vectors.append(singleDataItemEmbedding)
  return vectors

# Generate vectors for all Word2Vec models
print("Generating vectors for Word2Vec models...")

trainReviewVectors_w3 = getVectors(trainReviews, model_w3)
testReviewVectors_w3 = getVectors(testReviews, model_w3)
print("✓ Vectors generated for window=3")

trainReviewVectors_w5 = getVectors(trainReviews, model_w5)
testReviewVectors_w5 = getVectors(testReviews, model_w5)
print("✓ Vectors generated for window=5")

trainReviewVectors_w7 = getVectors(trainReviews, model_w7)
testReviewVectors_w7 = getVectors(testReviews, model_w7)
print("✓ Vectors generated for window=7")

Generating vectors for Word2Vec models...
✓ Vectors generated for window=3
✓ Vectors generated for window=5
✓ Vectors generated for window=7


In [9]:
# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

print("\nGenerating TF-IDF vectors...")

# Convert tokenized reviews back to strings for TF-IDF
trainReviewsStr = [' '.join(review) for review in trainReviews]
testReviewsStr = [' '.join(review) for review in testReviews]

# Create TF-IDF vectorizer with max_features to match embedding size for fair comparison
tfidf = TfidfVectorizer(max_features=embeddingsSize)
trainReviewVectors_tfidf = tfidf.fit_transform(trainReviewsStr).toarray()
testReviewVectors_tfidf = tfidf.transform(testReviewsStr).toarray()
print("✓ TF-IDF vectors generated")

print(f"\nAll vectorization methods ready!")
print(f"Vector dimension: {embeddingsSize}")


Generating TF-IDF vectors...
✓ TF-IDF vectors generated

All vectorization methods ready!
Vector dimension: 128


# Comparative Analysis: Multiple Vectorization Methods

This section compares **4 different vectorization methods**:
1. **Word2Vec (window=3)** - Smaller context window, captures local word relationships
2. **Word2Vec (window=5)** - Medium context window, balanced approach
3. **Word2Vec (window=7)** - Larger context window, captures broader context
4. **TF-IDF** - Traditional statistical method based on term frequency

Each method is tested with **4 machine learning classifiers**:
- **Naive Bayes (NB)** - Probabilistic classifier
- **Neural Network (NN)** - Multi-layer perceptron
- **Random Forest (RF)** - Ensemble method
- **K-Nearest Neighbors (KNN)** - Instance-based learning

**Total combinations tested: 16** (4 vectorization methods × 4 classifiers)

In [10]:
# Define evaluation metrics and helper functions
from sklearn.metrics import accuracy_score, precision_recall_fscore_support as score
from sklearn.metrics import f1_score, confusion_matrix

def printResults(y_true, y_predicted):
    """Display comprehensive evaluation metrics"""
    print("Accuracy= ", accuracy_score(y_true, y_predicted))
    
    # Print confusion matrix
    cm = confusion_matrix(y_true, y_predicted)
    print("Confusion Matrix:")
    print(cm)
    
    precision, recall, fscore, support = score(y_true, y_predicted)
    
    print('###########################################')
    print('precision: {}'.format(precision))  
    print('recall: {}'.format(recall))
    print('fscore: {}'.format(fscore))
    print('support: {}'.format(support))
    print('###########################################')
    
    print('Macro F1: ',f1_score(y_true, y_predicted, average='macro'))
    print('Micro F1: ', f1_score(y_true, y_predicted, average='micro'))

In [11]:
# Store all results for comparison
results_comparison = []

def runAllClassifiers(trainVectors, testVectors, method_name):
    """Run all 4 classifiers on given vectors and store results"""
    print(f"\n{'='*80}")
    print(f"RUNNING CLASSIFIERS FOR: {method_name}")
    print(f"{'='*80}\n")
    
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.neural_network import MLPClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    
    method_results = {'method': method_name}
    
    # 1. Naive Bayes
    print(f"{'='*80}")
    print(f"NAIVE BAYES - {method_name}")
    print(f"{'='*80}")
    scaler = MinMaxScaler()
    scaledTrainX = scaler.fit_transform(trainVectors)
    scaledTestX = scaler.transform(testVectors)
    
    clfNB = MultinomialNB()
    clfNB.fit(scaledTrainX, trainLabels)
    testLabelsPredicted = list(clfNB.predict(scaledTestX))
    nb_accuracy = accuracy_score(testLabels, testLabelsPredicted)
    nb_f1 = f1_score(testLabels, testLabelsPredicted, average='macro')
    printResults(testLabels, testLabelsPredicted)
    method_results['NB'] = {'accuracy': nb_accuracy, 'f1': nb_f1}
    
    # 2. Neural Network
    print(f"\n{'='*80}")
    print(f"NEURAL NETWORK - {method_name}")
    print(f"{'='*80}")
    clfMLP = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=500)
    clfMLP.fit(trainVectors, trainLabels)
    testLabelsPredicted = list(clfMLP.predict(testVectors))
    nn_accuracy = accuracy_score(testLabels, testLabelsPredicted)
    nn_f1 = f1_score(testLabels, testLabelsPredicted, average='macro')
    printResults(testLabels, testLabelsPredicted)
    method_results['NN'] = {'accuracy': nn_accuracy, 'f1': nn_f1}
    
    # 3. Random Forest
    print(f"\n{'='*80}")
    print(f"RANDOM FOREST - {method_name}")
    print(f"{'='*80}")
    clfRF = RandomForestClassifier(n_estimators=1000, random_state=42)
    clfRF.fit(trainVectors, trainLabels)
    testLabelsPredicted = list(clfRF.predict(testVectors))
    rf_accuracy = accuracy_score(testLabels, testLabelsPredicted)
    rf_f1 = f1_score(testLabels, testLabelsPredicted, average='macro')
    printResults(testLabels, testLabelsPredicted)
    method_results['RF'] = {'accuracy': rf_accuracy, 'f1': rf_f1}
    
    # 4. KNN
    print(f"\n{'='*80}")
    print(f"KNN - {method_name}")
    print(f"{'='*80}")
    clfKNN = KNeighborsClassifier(n_neighbors=3)
    clfKNN.fit(trainVectors, trainLabels)
    testLabelsPredicted = list(clfKNN.predict(testVectors))
    knn_accuracy = accuracy_score(testLabels, testLabelsPredicted)
    knn_f1 = f1_score(testLabels, testLabelsPredicted, average='macro')
    printResults(testLabels, testLabelsPredicted)
    method_results['KNN'] = {'accuracy': knn_accuracy, 'f1': knn_f1}
    
    results_comparison.append(method_results)
    return method_results

In [ ]:
# Run classifiers on Word2Vec (window=3)
runAllClassifiers(trainReviewVectors_w3, testReviewVectors_w3, "Word2Vec (window=3)")


RUNNING CLASSIFIERS FOR: Word2Vec (window=3)

NAIVE BAYES - Word2Vec (window=3)
Accuracy=  0.63648
Confusion Matrix:
[[2049 1055]
 [1217 1929]]
###########################################
precision: [0.62737293 0.64644772]
recall: [0.66011598 0.61315957]
fscore: [0.6433281  0.62936378]
support: [3104 3146]
###########################################
Macro F1:  0.6363459425682683
Micro F1:  0.63648

NEURAL NETWORK - Word2Vec (window=3)
Accuracy=  0.80752
Confusion Matrix:
[[2546  558]
 [ 645 2501]]
###########################################
precision: [0.79786901 0.81758745]
recall: [0.82023196 0.79497775]
fscore: [0.80889595 0.80612409]
support: [3104 3146]
###########################################
Macro F1:  0.8075100213195052
Micro F1:  0.80752

RANDOM FOREST - Word2Vec (window=3)


KeyboardInterrupt: 

In [ ]:
# Run classifiers on Word2Vec (window=5)
runAllClassifiers(trainReviewVectors_w5, testReviewVectors_w5, "Word2Vec (window=5)")

In [ ]:
# Run classifiers on Word2Vec (window=7)
runAllClassifiers(trainReviewVectors_w7, testReviewVectors_w7, "Word2Vec (window=7)")

In [ ]:
# Run classifiers on TF-IDF
runAllClassifiers(trainReviewVectors_tfidf, testReviewVectors_tfidf, "TF-IDF")

In [ ]:
# Display comprehensive results comparison
import pandas as pd

print("\n" + "="*100)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*100 + "\n")

# Create comparison DataFrame
comparison_data = []
for result in results_comparison:
    method = result['method']
    for classifier in ['NB', 'NN', 'RF', 'KNN']:
        comparison_data.append({
            'Vectorization Method': method,
            'Classifier': classifier,
            'Accuracy': f"{result[classifier]['accuracy']:.4f}",
            'Macro F1-Score': f"{result[classifier]['f1']:.4f}"
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "="*100)
print("BEST PERFORMERS")
print("="*100 + "\n")

# Find best combinations
best_accuracy_idx = comparison_df['Accuracy'].astype(float).idxmax()
best_f1_idx = comparison_df['Macro F1-Score'].astype(float).idxmax()

print(f"Best Accuracy: {comparison_df.loc[best_accuracy_idx, 'Vectorization Method']} + "
      f"{comparison_df.loc[best_accuracy_idx, 'Classifier']} = "
      f"{comparison_df.loc[best_accuracy_idx, 'Accuracy']}")

print(f"Best F1-Score: {comparison_df.loc[best_f1_idx, 'Vectorization Method']} + "
      f"{comparison_df.loc[best_f1_idx, 'Classifier']} = "
      f"{comparison_df.loc[best_f1_idx, 'Macro F1-Score']}")

# Average performance by vectorization method
print("\n" + "="*100)
print("AVERAGE PERFORMANCE BY VECTORIZATION METHOD")
print("="*100 + "\n")

avg_by_method = comparison_df.groupby('Vectorization Method').agg({
    'Accuracy': lambda x: f"{x.astype(float).mean():.4f}",
    'Macro F1-Score': lambda x: f"{x.astype(float).mean():.4f}"
})
print(avg_by_method)

# Average performance by classifier
print("\n" + "="*100)
print("AVERAGE PERFORMANCE BY CLASSIFIER")
print("="*100 + "\n")

avg_by_classifier = comparison_df.groupby('Classifier').agg({
    'Accuracy': lambda x: f"{x.astype(float).mean():.4f}",
    'Macro F1-Score': lambda x: f"{x.astype(float).mean():.4f}"
})
print(avg_by_classifier)

In [ ]:
import pickle
import os

# Create models directory if it doesn't exist
models_dir = 'saved_models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

print("\nSaving all models...\n")

# Save all Word2Vec models
model_w3.save(os.path.join(models_dir, 'word2vec_model_w3.model'))
print("✓ Word2Vec model (window=3) saved")

model_w5.save(os.path.join(models_dir, 'word2vec_model_w5.model'))
print("✓ Word2Vec model (window=5) saved")

model_w7.save(os.path.join(models_dir, 'word2vec_model_w7.model'))
print("✓ Word2Vec model (window=7) saved")

# Save TF-IDF vectorizer
with open(os.path.join(models_dir, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)
print("✓ TF-IDF vectorizer saved")

# Save embedding size and results comparison
with open(os.path.join(models_dir, 'embeddings_size.pkl'), 'wb') as f:
    pickle.dump(embeddingsSize, f)
print("✓ Embeddings size saved")

with open(os.path.join(models_dir, 'results_comparison.pkl'), 'wb') as f:
    pickle.dump(results_comparison, f)
print("✓ Results comparison saved")

print("\n✓ All models and results saved successfully in 'saved_models/' directory!")

In [ ]:
# Visualize results comparison
import matplotlib.pyplot as plt
import numpy as np

# Prepare data for visualization
methods = ['Word2Vec\n(w=3)', 'Word2Vec\n(w=5)', 'Word2Vec\n(w=7)', 'TF-IDF']
classifiers = ['NB', 'NN', 'RF', 'KNN']

# Create accuracy matrix
accuracy_matrix = []
f1_matrix = []

for result in results_comparison:
    accuracies = [result[clf]['accuracy'] for clf in classifiers]
    f1_scores = [result[clf]['f1'] for clf in classifiers]
    accuracy_matrix.append(accuracies)
    f1_matrix.append(f1_scores)

accuracy_matrix = np.array(accuracy_matrix).T
f1_matrix = np.array(f1_matrix).T

# Plot 1: Grouped Bar Chart for Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(methods))
width = 0.2

for i, clf in enumerate(classifiers):
    ax1.bar(x + i*width, accuracy_matrix[i], width, label=clf)

ax1.set_xlabel('Vectorization Method', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Accuracy Comparison: All Methods & Classifiers', fontsize=14, fontweight='bold')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(methods)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Grouped Bar Chart for F1-Score
for i, clf in enumerate(classifiers):
    ax2.bar(x + i*width, f1_matrix[i], width, label=clf)

ax2.set_xlabel('Vectorization Method', fontsize=12, fontweight='bold')
ax2.set_ylabel('Macro F1-Score', fontsize=12, fontweight='bold')
ax2.set_title('F1-Score Comparison: All Methods & Classifiers', fontsize=14, fontweight='bold')
ax2.set_xticks(x + width * 1.5)
ax2.set_xticklabels(methods)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved as 'results_comparison.png'")